In [0]:
%sql

--Setting up external storage locations
CREATE EXTERNAL LOCATION IF NOT EXISTS ukhsa_bronze
URL 'abfss://bronze@ukhsadev2026.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ukhsa-managed-identity`);
 
CREATE EXTERNAL LOCATION IF NOT EXISTS ukhsa_silver
URL 'abfss://silver@ukhsadev2026.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ukhsa-managed-identity`);
 
CREATE EXTERNAL LOCATION IF NOT EXISTS ukhsa_gold
URL 'abfss://gold@ukhsadev2026.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ukhsa-managed-identity`);

"""
UKHSA Bronze layer ingestion.

API structure:

GET /themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/
      geography_types/{geography_type}/geographies/{geography}/
      metrics/{metric}

This script:
1. Discovers the available geographies for a given geography_type
2. Loops topic/metric/geography combinations.
3. Paginates each call with page_size=365 and follows 'next'.
4. Writes raw JSON pages to the Bronze path, one file per page.
"""

In [0]:
import requests
import json
import datetime
import time

#UKHSA API base URL
BASE = "https://api.ukhsa-dashboard.data.gov.uk"

# Bronze storage root. Writes via the ukhsa_bronze external location,
# created against the bronze container using the ukhsa-managed-identity
# storage credential.
bronze_path = "abfss://bronze@ukhsadev2026.dfs.core.windows.net"

# All API parameters for all the calls. 
request_configs = [
    {
        "theme": "infectious_disease",
        "sub_theme": "respiratory",
        "topic": "COVID-19",
        "metric": "COVID-19_cases_casesByDay",
    },
]

GEOGRAPHY_TYPE = "Lower Tier Local Authority" 


def discover_geography_types(theme, sub_theme, topic):
    """List geography_types available for a topic."""
    url = f"{BASE}/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/geography_types"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()


def discover_geographies(theme, sub_theme, topic, geography_type):
    """List every geography name available under a geography_type."""
    url = (
        f"{BASE}/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}"
        f"/geography_types/{geography_type}/geographies"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    # Data is a list of dictionaries
    # [{"name":"Nation", "link": "https://api.ukhsa-dashboard.data.gov.uk/themes/infectious_disease/sub_themes/respiratory/topics/Influenza/geography_types/Nation"" }, ....]
    return data


def fetch_and_save(theme, sub_theme, topic, geography_type, geography, metric):
    url = (
        f"{BASE}/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}"
        f"/geography_types/{geography_type}/geographies/{geography}/metrics/{metric}"
    )
    params = {"page_size": 365}
    date_stamp = datetime.datetime.now().strftime("%Y%m%d")
    page_num = 1
    current_url = url

    while current_url:
        resp = requests.get(current_url, params=params if page_num == 1 else None)

        if resp.status_code == 200:
            data = resp.json()

            safe_geo = geography.replace(" ", "_").replace("/", "-")
            file_path = (
                f"{bronze_path}/ukhsa/{topic}/{metric}/{safe_geo}/"
                f"{date_stamp}_page_{page_num}.json"
            )
            dbutils.fs.put(file_path, json.dumps(data), overwrite=True)

            current_url = data.get("next")
            page_num += 1
            params = None
        elif resp.status_code == 404:
            # If the geography/metric combination doesn't exist for this topic, skip quietly
            print(f"No data: {topic} / {metric} / {geography}")
            break
        else:
            print(f"Error {resp.status_code} on {topic}/{metric}/{geography}: {resp.text}")
            break

        time.sleep(0.1) 

In [0]:
#Main Application Loop

for config in request_configs:
    geographies = discover_geographies(
        config["theme"], config["sub_theme"], config["topic"], GEOGRAPHY_TYPE
    )
    # Response is a plain list of {"name": ..., "link": ...} entries
    geo_names = [g["name"] for g in geographies]

    for geo in geo_names:
        fetch_and_save(
            config["theme"],
            config["sub_theme"],
            config["topic"],
            GEOGRAPHY_TYPE,
            geo,
            config["metric"],
        )

print("Ingestion complete.")

In [0]:
#Checking that the ingestion was successful
display(dbutils.fs.ls("abfss://bronze@ukhsadev2026.dfs.core.windows.net/ukhsa/COVID-19/"))
#By local authority
display(dbutils.fs.ls("abfss://bronze@ukhsadev2026.dfs.core.windows.net/ukhsa/COVID-19/COVID-19_cases_casesByDay/"))
#Specifyign Leeds
display(dbutils.fs.ls("abfss://bronze@ukhsadev2026.dfs.core.windows.net/ukhsa/COVID-19/COVID-19_cases_casesByDay/Leeds/"))